# Tool Selection: Three Independent Agents

The same question goes to three **independent** agents. Each trims this demo's
tool pool (`semantic_tools.ALL_TOOLS`, backed by Neo4j for the hotel tools) to
the relevant tool with a before-model hook, but each uses a different selector:

1. **FAISS** - vector nearest-neighbour over tool-description embeddings.
2. **strands-decider 2B** - the open-source Strands decision model, local.
3. **Jev** - TypeSafe's decision model, hosted via OpenRouter.

They share nothing: FAISS does not touch the decider, the decider does not touch
FAISS. Each section imports only what that agent needs.

**Setup:** running Neo4j (demo 01), AWS creds for Bedrock, `pip install
strands-decider`, `OPENROUTER_API_KEY` for Jev.

## Shared pieces: the hook, the tools, the question

In [ ]:
import time
from strands import Agent
from strands.models import BedrockModel
from strands.hooks import HookProvider, HookRegistry, BeforeModelCallEvent
from semantic_tools import ALL_TOOLS

BY_NAME = {t.tool_name: t for t in ALL_TOOLS}
CRITERIA = {t.tool_name: t.tool_spec["description"] for t in ALL_TOOLS}
AGENT_MODEL = BedrockModel(model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0", region_name="us-east-1")
PROMPT = "You are a travel agent that helps people with hotels. Keep answers concise."
QUESTION = "How much is a room at the Marriott?"

class ToolFilterHook(HookProvider):
    """Before the model is called, pick the tool(s) and swap them into the agent."""
    def __init__(self, selector):
        self.selector = selector; self.pick_ms = 0.0
    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeModelCallEvent, self._before_model)
    async def _before_model(self, event: BeforeModelCallEvent) -> None:
        q = ""
        for m in reversed(event.agent.messages):
            if m.get("role") == "user":
                q = "".join(b.get("text", "") for b in m.get("content", []) if "text" in b); break
        if not q: return
        t0 = time.perf_counter(); chosen = await self.selector(q); self.pick_ms = (time.perf_counter()-t0)*1000
        reg = event.agent.tool_registry
        reg.registry.clear(); reg.dynamic_tools.clear()
        for t in chosen: reg.register_tool(t)

async def ask(agent, hook, label):
    t0 = time.perf_counter(); result = await agent.invoke_async(QUESTION); total = (time.perf_counter()-t0)*1000
    called = list(result.metrics.tool_metrics.keys()) if result.metrics else []
    tok = result.metrics.accumulated_usage["totalTokens"] if result.metrics else 0
    print(f"[{label}] tool={called} tokens={tok} pick={hook.pick_ms:.0f}ms total={total:.0f}ms")
    print(f"[{label}] {str(result)[:160]}")

print(f"{len(ALL_TOOLS)} tools. Question: {QUESTION!r}")

## Agent 1: FAISS (vector)

Independent agent. Imports only FAISS + boto3 for embeddings.

In [ ]:
import boto3, json, numpy as np, faiss
_br = boto3.client("bedrock-runtime", region_name="us-east-1")
def _embed(text):
    r = _br.invoke_model(modelId="amazon.titan-embed-text-v2:0",
        body=json.dumps({"inputText": text}), contentType="application/json", accept="application/json")
    return np.array(json.loads(r["body"].read())["embedding"], dtype="float32")
_names = list(CRITERIA)
_vecs = np.array([_embed(d) for d in CRITERIA.values()])
_index = faiss.IndexFlatL2(_vecs.shape[1]); _index.add(_vecs)
async def faiss_select(query, k=3):
    _, idx = _index.search(np.array([_embed(query)]), k)
    return [BY_NAME[_names[i]] for i in idx[0]]

faiss_hook = ToolFilterHook(faiss_select)
faiss_agent = Agent(model=AGENT_MODEL, tools=ALL_TOOLS, system_prompt=PROMPT, hooks=[faiss_hook], callback_handler=None)
await ask(faiss_agent, faiss_hook, "faiss")

## Agent 2: strands-decider 2B (local)

Independent agent. The 2B weights download on first run and are cached after.

In [ ]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1"); os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
from strands_decider.infer import load_engine
from strands_decider.schema import ChoiceQuestion
_engine = load_engine("StrandsAgents/strands-decider-2B-hobson-v19", device="cpu")
async def decider_select(query):
    resp = _engine.ask(query, {"tool": ChoiceQuestion(instructions="Which tool answers this request?", criteria=CRITERIA)})
    return [BY_NAME[resp.answers["tool"].choice]]

decider_hook = ToolFilterHook(decider_select)
decider_agent = Agent(model=AGENT_MODEL, tools=ALL_TOOLS, system_prompt=PROMPT, hooks=[decider_hook], callback_handler=None)
await ask(decider_agent, decider_hook, "decider")

## Agent 3: Jev (hosted, OpenRouter)

Independent agent. Imports only the TypeSafe SDK.

In [ ]:
import os
from typesafe_sdk import AsyncTypeSafeClient, Choice
_jev = AsyncTypeSafeClient(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api", model="~typesafe/jev-latest")
async def jev_select(query):
    resp = await _jev.system_one(state=query, questions={"tool": Choice(instructions="Which tool answers this request?", criteria=CRITERIA)})
    return [BY_NAME[resp.answers["tool"].choice]]

jev_hook = ToolFilterHook(jev_select)
jev_agent = Agent(model=AGENT_MODEL, tools=ALL_TOOLS, system_prompt=PROMPT, hooks=[jev_hook], callback_handler=None)
await ask(jev_agent, jev_hook, "jev")

## Note on running

FAISS (Agent 1) and torch, which strands-decider pulls in (Agent 2), each ship
their own OpenMP runtime and can clash in one kernel. The agents are independent,
so run each section on its own, or use `chat.py` / `chat_decider.py`, which run
one agent per process.